# 01 — Baselines

**Objetivo.** Establecer el suelo de referencia validado a 252 días. Primer entregable que asegura el aprobado (RMSE < 75 000).

**Decisión tomada.** Baselines antes que cualquier NN. Si algún baseline ya clava un índice (especialmente B), documenta qué nivel hay que batir en los demás.

**Qué hace.** `eval_all_baselines` para los 6 índices → tabla de RMSE (flat/drift/random_walk). Guarda `results/baselines.json`.

**Qué NO hace.** No entrena redes. No usa datos auxiliares. No escribe ningún `results/index_X.json` — cada notebook de índice (03-08) es el único responsable de escribir su propio JSON.

**Inputs.** `data/train_indices.csv`

**Output esperado.** `results/baselines.json` — dict `{índice: {flat: RMSE, drift: RMSE, random_walk: RMSE}}`.

## 0. GPU workaround + imports

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data, eval_all_baselines,
    backtest_autoregressive, baseline_flat, baseline_drift, baseline_random_walk,
    plot_rollout, plot_rmse_by_index,
    DATA_DIR, VAL_DAYS, V_IN_SHARED, INDEX_COLS
)

## 1. Carga de datos

In [ ]:
data = load_hackathon_data(DATA_DIR)
idx  = data['train_indices']

## 2. Backtest de los 3 baselines para los 6 índices

`eval_all_baselines` ejecuta flat/drift/random_walk con `backtest_autoregressive` internamente. El resultado es un DataFrame con RMSE por índice y baseline, ordenado de mejor a peor RMSE medio.

In [ ]:
df_baselines = eval_all_baselines(data, val_days=VAL_DAYS, v_in=V_IN_SHARED)
print(df_baselines.round(0))

## 3. Visualización: RMSE por índice y baseline

In [ ]:
# Tabla en formato dict para plot_rmse_by_index
rmse_dict = {
    baseline: {col: df_baselines.loc[baseline, col]
               for col in INDEX_COLS if col in df_baselines.columns}
    for baseline in df_baselines.index
}
plot_rmse_by_index(rmse_dict, title='RMSE por baseline (252d backtest)')

## 4. Rollout visual del mejor baseline por índice

Para cada índice: identificar el baseline ganador y graficar su rollout vs precios reales.

In [ ]:
# Para cada índice: correr el mejor baseline manualmente y graficar
for col in INDEX_COLS:
    if col not in idx.columns:
        continue
    serie = idx[col].dropna().values
    n_train = len(serie) - VAL_DAYS
    train_s = serie[:n_train]

    # Mejor baseline según df_baselines
    best_bl = df_baselines[col].idxmin()   # nombre del baseline con menor RMSE

    if best_bl == 'flat':
        preds = baseline_flat(train_s, VAL_DAYS)
    elif best_bl == 'drift':
        preds = baseline_drift(train_s, VAL_DAYS)
    else:
        preds = baseline_random_walk(train_s, VAL_DAYS)

    plot_rollout(serie, preds, index_name=f'{col} — {best_bl}', val_days=VAL_DAYS)

## 5. Decisión y guardado de baselines.json

**REGLA:** Este notebook escribe SOLO `results/baselines.json`. Cada `results/index_X.json` lo escribe EXCLUSIVAMENTE el notebook de ese índice (03-08), incluso si el approach ganador es un baseline.

In [ ]:
# Construir dict {índice: {flat: RMSE, drift: RMSE, random_walk: RMSE}}
baselines_out = {}
for col in INDEX_COLS:
    if col in df_baselines.columns:
        baselines_out[col] = {
            'flat':        float(df_baselines.loc['flat',        col]),
            'drift':       float(df_baselines.loc['drift',       col]),
            'random_walk': float(df_baselines.loc['random_walk', col]),
        }

os.makedirs('results', exist_ok=True)
with open('results/baselines.json', 'w') as f:
    json.dump(baselines_out, f, indent=2)

print('Guardado: results/baselines.json')
print(json.dumps(baselines_out, indent=2))

## 6. Resumen

Rellenar tras ejecutar.

| Índice | Mejor baseline | RMSE | ¿Basta para aprobar? |
|--------|---------------|------|----------------------|
| A | ? | ? | ? |
| B | flat | ? | ? |
| C | ? | ? | ? |
| D | ? | ? | ? |
| E | ? | ? | ? |
| F | ? | ? | ? |

RMSE medio global de los baselines: `?`  
Umbral de aprobado: 75 000. ¿Pasamos con solo baselines? `?`